# NB04: Pangenome Annotation Evidence (Tier 2)

**Purpose**: Extract EC and KEGG reaction annotations from pangenome tables:
1. eggNOG EC (25.9M gene clusters with EC, 27.8% of 93.6M)
2. eggNOG KEGG_Reaction (20.3M gene clusters with R-numbers)
3. bakta EC (19.0M from annotations + 15.2M from db_xrefs)

These map gene clusters (not individual proteins). Coverage stats show how many
balanced reactions are reachable; gene-cluster-to-protein expansion deferred.

**Key**: EC and KEGG_Reaction fields can be comma-separated multivalues; must explode.

**Output**: `pangenome_gc_ec.parquet`, `pangenome_gc_kegg.parquet`

**Requires**: BERDL JupyterHub (Spark session), NB02 bridge tables

In [1]:
import os, re
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()
spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
kegg_bridge = pd.read_parquet(f'{DATA_DIR}/kegg_to_reaction.parquet')
bridge_ecs = set(ec_bridge['ec'])
bridge_keggs = set(kegg_bridge['kegg_reaction'])

print(f'EC bridge:   {len(ec_bridge):,} mappings, {len(bridge_ecs):,} ECs')
print(f'KEGG bridge: {len(kegg_bridge):,} mappings, {len(bridge_keggs):,} R-numbers')
print(f'Spark session ready.')

EC bridge:   22,823 mappings, 6,100 ECs
KEGG bridge: 6,851 mappings, 6,354 R-numbers
Spark session ready.


## 1. eggNOG EC Annotations

EC from `eggnog_mapper_annotations.EC` (comma-separated, ~26M non-null rows).
The `query_name` column is the gene_cluster_id.

In [2]:
eggnog_ec_spark = spark.sql("""
    SELECT query_name AS gene_cluster_id, EC
    FROM kbase_ke_pangenome.eggnog_mapper_annotations
    WHERE EC IS NOT NULL AND EC != '' AND EC != '-'
""")

eggnog_ec_count = eggnog_ec_spark.count()
print(f'eggNOG rows with EC: {eggnog_ec_count:,}')

eggnog_ec_sample = eggnog_ec_spark.limit(10).toPandas()
print(f'\nSample EC values:')
for _, row in eggnog_ec_sample.iterrows():
    print(f'  {row["gene_cluster_id"][:40]:40s} EC={row["EC"]}')

eggNOG rows with EC: 25,962,995



Sample EC values:
  JAJWPL010000128.1_5                      EC=2.5.1.47
  CADAOQ010000015.1_18                     EC=1.1.1.122
  CADAOQ010000013.1_26                     EC=1.1.1.193,3.5.4.26
  QUMF01000051.1_12                        EC=2.7.7.6
  QUMF01000001.1_140                       EC=2.5.1.72
  USTH01000012.1_6                         EC=3.4.21.105
  NZ_QSRL01000001.1_113                    EC=2.3.1.46
  CAJMLP010000023.1_9                      EC=3.1.1.41
  CAJMLP010000011.1_33                     EC=4.99.1.3
  NZ_QJKC01000019.1_45                     EC=5.1.1.8


In [3]:
from pyspark.sql import functions as F

eggnog_ec_exploded = eggnog_ec_spark.select(
    'gene_cluster_id',
    F.explode(F.split(F.col('EC'), ',')).alias('ec')
).filter(F.col('ec') != '').filter(F.col('ec') != '-')

eggnog_ec_unique = eggnog_ec_exploded.select('ec').distinct().toPandas()
eggnog_ecs = set(eggnog_ec_unique['ec'])

print(f'eggNOG unique ECs after explode: {len(eggnog_ecs):,}')
print(f'  Matching bridge: {len(eggnog_ecs & bridge_ecs):,}')
print(f'  Not in bridge:   {len(eggnog_ecs - bridge_ecs):,}')

eggnog_ec_rxns = set(ec_bridge[ec_bridge['ec'].isin(eggnog_ecs)]['rxn_bare'])
print(f'  Balanced reactions reachable: {len(eggnog_ec_rxns):,}')

eggNOG unique ECs after explode: 3,820
  Matching bridge: 3,286
  Not in bridge:   534
  Balanced reactions reachable: 11,778


In [4]:
eggnog_ec_filtered = eggnog_ec_exploded.filter(
    eggnog_ec_exploded.ec.isin(list(bridge_ecs))
).select('gene_cluster_id', 'ec').distinct()

eggnog_ec_pairs = eggnog_ec_filtered.toPandas()
eggnog_ec_pairs['channel'] = 'eggnog_ec'

print(f'eggNOG EC gene_cluster-EC pairs (bridge-matched): {len(eggnog_ec_pairs):,}')
print(f'  Unique gene clusters: {eggnog_ec_pairs.gene_cluster_id.nunique():,}')
print(f'  Unique ECs:           {eggnog_ec_pairs.ec.nunique():,}')

eggNOG EC gene_cluster-EC pairs (bridge-matched): 26,419,085


  Unique gene clusters: 22,641,907


  Unique ECs:           3,286


## 2. eggNOG KEGG Reaction Annotations

KEGG R-numbers from `KEGG_Reaction` column (comma-separated).

In [5]:
eggnog_kegg_spark = spark.sql("""
    SELECT query_name AS gene_cluster_id, KEGG_Reaction
    FROM kbase_ke_pangenome.eggnog_mapper_annotations
    WHERE KEGG_Reaction IS NOT NULL AND KEGG_Reaction != '' AND KEGG_Reaction != '-'
""")

eggnog_kegg_count = eggnog_kegg_spark.count()
print(f'eggNOG rows with KEGG_Reaction: {eggnog_kegg_count:,}')

eggnog_kegg_sample = eggnog_kegg_spark.limit(10).toPandas()
print(f'\nSample KEGG_Reaction values:')
for _, row in eggnog_kegg_sample.iterrows():
    print(f'  {row["gene_cluster_id"][:40]:40s} KEGG={row["KEGG_Reaction"]}')

eggNOG rows with KEGG_Reaction: 20,286,023



Sample KEGG_Reaction values:
  JAJWPL010000128.1_5                      KEGG=R00897,R03601,R04859
  CADAOQ010000015.1_18                     KEGG=R07675,R08926
  CADAOQ010000013.1_26                     KEGG=R03458,R03459
  QUMF01000051.1_12                        KEGG=R00435,R00441,R00442,R00443
  QUMF01000001.1_140                       KEGG=R04292
  NZ_QSRL01000001.1_113                    KEGG=R01777
  CAJMLP010000023.1_9                      KEGG=R03062
  CAJMLP010000011.1_33                     KEGG=R05807
  NZ_QJKC01000019.1_45                     KEGG=R03296
  NZ_LNQU01000003.1_80                     KEGG=R01055


In [6]:
eggnog_kegg_exploded = eggnog_kegg_spark.select(
    'gene_cluster_id',
    F.explode(F.split(F.col('KEGG_Reaction'), ',')).alias('kegg_reaction')
).filter(F.col('kegg_reaction') != '').filter(F.col('kegg_reaction') != '-')

eggnog_kegg_unique = eggnog_kegg_exploded.select('kegg_reaction').distinct().toPandas()
eggnog_keggs = set(eggnog_kegg_unique['kegg_reaction'])

print(f'eggNOG unique KEGG R-numbers after explode: {len(eggnog_keggs):,}')
print(f'  Matching bridge: {len(eggnog_keggs & bridge_keggs):,}')
print(f'  Not in bridge:   {len(eggnog_keggs - bridge_keggs):,}')

eggnog_kegg_rxns = set(kegg_bridge[kegg_bridge['kegg_reaction'].isin(eggnog_keggs)]['rxn_bare'])
print(f'  Balanced reactions reachable: {len(eggnog_kegg_rxns):,}')

eggNOG unique KEGG R-numbers after explode: 4,969
  Matching bridge: 3,295
  Not in bridge:   1,674
  Balanced reactions reachable: 3,529


In [7]:
eggnog_kegg_filtered = eggnog_kegg_exploded.filter(
    eggnog_kegg_exploded.kegg_reaction.isin(list(bridge_keggs))
).select('gene_cluster_id', 'kegg_reaction').distinct()

eggnog_kegg_gc_count = eggnog_kegg_filtered.select('gene_cluster_id').distinct().count()
eggnog_kegg_r_count = eggnog_kegg_filtered.select('kegg_reaction').distinct().count()
eggnog_kegg_pair_count = eggnog_kegg_filtered.count()

print(f'eggNOG KEGG gene_cluster-reaction pairs (bridge-matched): {eggnog_kegg_pair_count:,}')
print(f'  Unique gene clusters:  {eggnog_kegg_gc_count:,}')
print(f'  Unique KEGG R-numbers: {eggnog_kegg_r_count:,}')

eggNOG KEGG gene_cluster-reaction pairs (bridge-matched): 35,570,781
  Unique gene clusters:  15,486,765
  Unique KEGG R-numbers: 3,295


## 3. bakta EC Annotations

EC from two sources:
- `bakta_annotations.ec` (19.0M rows with EC)
- `bakta_db_xrefs WHERE db='EC'` (15.2M rows)

bakta uses `gene_cluster_id` directly.

In [8]:
bakta_ec_ann = spark.sql("""
    SELECT gene_cluster_id, ec
    FROM kbase_ke_pangenome.bakta_annotations
    WHERE ec IS NOT NULL AND ec != ''
""")

bakta_ann_count = bakta_ec_ann.count()
print(f'bakta_annotations rows with ec: {bakta_ann_count:,}')

bakta_ann_sample = bakta_ec_ann.limit(10).toPandas()
print(f'\nSample bakta_annotations EC values:')
for _, row in bakta_ann_sample.iterrows():
    print(f'  {row["gene_cluster_id"][:40]:40s} ec={row["ec"]}')

bakta_annotations rows with ec: 19,040,536



Sample bakta_annotations EC values:
  CAKMWB010000001.1_1097                   ec=5.3.1.27
  CAKMWB010000001.1_1107                   ec=5.1.3.4
  CAKMWB010000001.1_1213                   ec=2.8.4.5
  CAKMWB010000001.1_1303                   ec=3.1.21.5
  CAKMWB010000001.1_160                    ec=2.4.1.25
  CAKMWB010000001.1_261                    ec=1.8.4.10;1.8.4.8
  CAKMWB010000001.1_275                    ec=3.1.21.5
  CAKMWB010000001.1_29                     ec=2.4.2.8
  CAKMWB010000001.1_317                    ec=3.1.3.16
  CAKMWB010000001.1_318                    ec=2.1.1.192


In [9]:
bakta_ec_xrefs = spark.sql("""
    SELECT gene_cluster_id, accession AS ec
    FROM kbase_ke_pangenome.bakta_db_xrefs
    WHERE db = 'EC'
""")

bakta_xref_count = bakta_ec_xrefs.count()
print(f'bakta_db_xrefs EC rows: {bakta_xref_count:,}')

bakta_xref_sample = bakta_ec_xrefs.limit(10).toPandas()
print(f'\nSample bakta_db_xrefs EC values:')
for _, row in bakta_xref_sample.iterrows():
    print(f'  {row["gene_cluster_id"][:40]:40s} ec={row["ec"]}')

bakta_db_xrefs EC rows: 15,177,756



Sample bakta_db_xrefs EC values:
  URCW01000033.1_19                        ec=3.5.1.28
  URCW01000033.1_2                         ec=7.1.2.2
  URCW01000033.1_5                         ec=7.1.2.2
  URCW01000033.1_5                         ec=7.2.2.1
  URCW01000033.1_6                         ec=7.1.2.2
  URCW01000033.1_9                         ec=7.1.2.2
  URCW01000036.1_9                         ec=2.1.1.207
  URCW01000038.1_13                        ec=2.8.4.5
  URCW01000038.1_14                        ec=2.4.2.9
  URCW01000042.1_9                         ec=2.8.1.7


In [10]:
bakta_ec_all = bakta_ec_ann.union(bakta_ec_xrefs)

bakta_ec_unique_ecs = bakta_ec_all.select('ec').distinct().toPandas()
bakta_ecs = set(bakta_ec_unique_ecs['ec'])

print(f'bakta unique ECs (combined): {len(bakta_ecs):,}')
print(f'  Matching bridge: {len(bakta_ecs & bridge_ecs):,}')
print(f'  Not in bridge:   {len(bakta_ecs - bridge_ecs):,}')

bakta_ec_rxns = set(ec_bridge[ec_bridge['ec'].isin(bakta_ecs)]['rxn_bare'])
print(f'  Balanced reactions reachable: {len(bakta_ec_rxns):,}')

bakta unique ECs (combined): 5,557
  Matching bridge: 2,926
  Not in bridge:   2,631
  Balanced reactions reachable: 11,907


In [11]:
bakta_ec_filtered = bakta_ec_all.filter(
    bakta_ec_all.ec.isin(list(bridge_ecs))
).select('gene_cluster_id', 'ec').distinct()

bakta_ec_pairs = bakta_ec_filtered.toPandas()
bakta_ec_pairs['channel'] = 'bakta_ec'

print(f'bakta EC gene_cluster-EC pairs (bridge-matched): {len(bakta_ec_pairs):,}')
print(f'  Unique gene clusters: {bakta_ec_pairs.gene_cluster_id.nunique():,}')
print(f'  Unique ECs:           {bakta_ec_pairs.ec.nunique():,}')

bakta EC gene_cluster-EC pairs (bridge-matched): 15,432,776


  Unique gene clusters: 14,549,018


  Unique ECs:           2,926


## 4. Combine & Coverage Summary

In [12]:
pangenome_ec = pd.concat([
    eggnog_ec_pairs[['gene_cluster_id', 'ec', 'channel']],
    bakta_ec_pairs[['gene_cluster_id', 'ec', 'channel']]
], ignore_index=True).drop_duplicates(subset=['gene_cluster_id', 'ec', 'channel'])

pangenome_ec_dedup = pangenome_ec.groupby(['gene_cluster_id', 'ec'])['channel'].agg(
    channels=lambda x: ','.join(sorted(set(x))),
    n_channels='nunique'
).reset_index()

print(f'Pangenome EC evidence (combined):')
print(f'  Gene_cluster-EC pairs: {len(pangenome_ec_dedup):,}')
print(f'  Unique gene clusters:  {pangenome_ec_dedup.gene_cluster_id.nunique():,}')
print(f'  Unique ECs:            {pangenome_ec_dedup.ec.nunique():,}')

print(f'\nChannel overlap:')
combo = pangenome_ec_dedup['channels'].value_counts()
for ch, ct in combo.items():
    print(f'  {ch}: {ct:,}')

Pangenome EC evidence (combined):
  Gene_cluster-EC pairs: 30,180,448


  Unique gene clusters:  25,367,594


  Unique ECs:            3,783

Channel overlap:
  eggnog_ec: 14,747,672
  bakta_ec,eggnog_ec: 11,671,413
  bakta_ec: 3,761,363


In [13]:
balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

all_pangenome_ecs = set(pangenome_ec_dedup['ec'])
all_pangenome_rxns = set(ec_bridge[ec_bridge['ec'].isin(all_pangenome_ecs)]['rxn_bare'])
all_kegg_rxns = eggnog_kegg_rxns
tier2_rxns = all_pangenome_rxns | all_kegg_rxns

print(f'Balanced reactions reached by Tier 2:')
print(f'  eggNOG EC:    {len(eggnog_ec_rxns):,} / {len(balanced_ids):,} ({100*len(eggnog_ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  eggNOG KEGG:  {len(eggnog_kegg_rxns):,} / {len(balanced_ids):,} ({100*len(eggnog_kegg_rxns)/len(balanced_ids):.1f}%)')
print(f'  bakta EC:     {len(bakta_ec_rxns):,} / {len(balanced_ids):,} ({100*len(bakta_ec_rxns)/len(balanced_ids):.1f}%)')
print(f'  Combined:     {len(tier2_rxns):,} / {len(balanced_ids):,} ({100*len(tier2_rxns)/len(balanced_ids):.1f}%)')

print(f'\nAdditive value:')
print(f'  eggNOG KEGG beyond eggNOG EC: {len(eggnog_kegg_rxns - eggnog_ec_rxns):,} additional reactions')
print(f'  bakta beyond eggNOG:          {len(bakta_ec_rxns - eggnog_ec_rxns - eggnog_kegg_rxns):,} additional reactions')

Balanced reactions reached by Tier 2:
  eggNOG EC:    11,778 / 34,343 (34.3%)
  eggNOG KEGG:  3,529 / 34,343 (10.3%)
  bakta EC:     11,907 / 34,343 (34.7%)
  Combined:     14,705 / 34,343 (42.8%)

Additive value:
  eggNOG KEGG beyond eggNOG EC: 437 additional reactions
  bakta beyond eggNOG:          2,490 additional reactions


In [14]:
pangenome_ec_dedup.to_parquet(f'{DATA_DIR}/pangenome_gc_ec.parquet', index=False)
print(f'Saved {len(pangenome_ec_dedup):,} gene_cluster-EC pairs to pangenome_gc_ec.parquet')
print(f'  (KEGG gene_cluster pairs not saved locally -- {eggnog_kegg_pair_count:,} pairs computed on Spark)')

Saved 30,180,448 gene_cluster-EC pairs to pangenome_gc_ec.parquet
  (KEGG gene_cluster pairs not saved locally -- 35,570,781 pairs computed on Spark)


## 5. Summary

In [15]:
print('=' * 60)
print('NB04 TIER 2 EVIDENCE SUMMARY')
print('=' * 60)
print(f'\nChannels:')
print(f'  1. eggNOG EC:   {eggnog_ec_pairs.gene_cluster_id.nunique():,} gene clusters -> {len(eggnog_ec_rxns):,} reactions')
print(f'  2. eggNOG KEGG: {eggnog_kegg_gc_count:,} gene clusters -> {len(eggnog_kegg_rxns):,} reactions')
print(f'  3. bakta EC:    {bakta_ec_pairs.gene_cluster_id.nunique():,} gene clusters -> {len(bakta_ec_rxns):,} reactions')
print(f'\nCombined Tier 2: {len(tier2_rxns):,} / {len(balanced_ids):,} balanced reactions ({100*len(tier2_rxns)/len(balanced_ids):.1f}%)')
print(f'\nSaved:')
print(f'  pangenome_gc_ec.parquet ({len(pangenome_ec_dedup):,} pairs)')
print(f'\nNext: NB05 -- curated & specialized evidence (Tiers 3-4)')

NB04 TIER 2 EVIDENCE SUMMARY

Channels:


  1. eggNOG EC:   22,641,907 gene clusters -> 11,778 reactions
  2. eggNOG KEGG: 15,486,765 gene clusters -> 3,529 reactions


  3. bakta EC:    14,549,018 gene clusters -> 11,907 reactions

Combined Tier 2: 14,705 / 34,343 balanced reactions (42.8%)

Saved:
  pangenome_gc_ec.parquet (30,180,448 pairs)

Next: NB05 -- curated & specialized evidence (Tiers 3-4)
